# Car Price Prediction with Machine Learning
Objective: predict used-car selling price using age, mileage, fuel type, transmission, and brand.


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
df=pd.read_csv("data/car_details.csv"); df.columns=df.columns.str.strip().str.lower().str.replace(" ","_")
df=df.drop_duplicates(); df["fuel"]=df["fuel"].astype(str).str.strip().str.lower(); df["transmission"]=df["transmission"].astype(str).str.strip().str.lower()
df["car_age"]=pd.Timestamp.now().year-df["year"]; df["brand"]=df["name"].astype(str).str.split().str[0]
print("shape:",df.shape); print("nulls:",df.isnull().sum()); display(df.head())

In [ ]:
sns.histplot(df.selling_price,kde=True); plt.title("Selling price distribution"); plt.show()
sns.boxplot(data=df,x="fuel",y="selling_price"); plt.xticks(rotation=30); plt.show()
sns.scatterplot(data=df,x="car_age",y="selling_price",hue="transmission"); plt.show()
num=df.select_dtypes("number"); sns.heatmap(num.corr(),annot=True,cmap="vlag"); plt.title("Feature correlation"); plt.show()

In [ ]:
features=["car_age","km_driven","fuel","seller_type","transmission","owner","brand"]; X=df[features]; y=df.selling_price
cat=["fuel","seller_type","transmission","owner","brand"]; num=["car_age","km_driven"]
prep=ColumnTransformer([("num",SimpleImputer(strategy="median"),num),("cat",Pipeline([("impute",SimpleImputer(strategy="most_frequent")),("onehot",OneHotEncoder(handle_unknown="ignore"))]),cat)])
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.2,random_state=42)
models={"Linear Regression":LinearRegression(),"Random Forest":RandomForestRegressor(n_estimators=150,random_state=42,n_jobs=-1)}; scores={}
for name,est in models.items():
 pipe=Pipeline([("prep",prep),("model",est)]); pipe.fit(Xtr,ytr); pred=pipe.predict(Xte); scores[name]={'MAE':mean_absolute_error(yte,pred),'RMSE':mean_squared_error(yte,pred)**.5,'R2':r2_score(yte,pred)}; print(name,scores[name])
best_name=max(scores,key=lambda k:scores[k]['R2']); best=models[best_name]; print("Best model:",best_name)

In [ ]:
if best_name=="Random Forest":
 names=best_pipe_names=[]
 # permutation importance works with the full preprocessing pipeline
 from sklearn.inspection import permutation_importance
 fitted=Pipeline([("prep",prep),("model",best)]).fit(Xtr,ytr)
 imp=permutation_importance(fitted,Xte,yte,n_repeats=5,random_state=42)
 order=np.argsort(imp.importances_mean)[-10:]
 plt.barh(range(len(order)),imp.importances_mean[order]); plt.yticks(range(len(order)),np.array(features)[order]); plt.title("Permutation feature importance"); plt.show()